In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0" pillow pandas requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import json
import math
import re
from pathlib import Path

import requests
import torch
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoProcessor, GenerationConfig,
    Qwen3VLForConditionalGeneration
)

In [4]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
DATA = ROOT / "data/grounding20"
IMAGES = DATA / "images"
OUTPUTS = ROOT / "outputs/grounding"
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=False)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="cuda:0",
    attn_implementation="sdpa"
).eval()

GEN = GenerationConfig(
    max_new_tokens=64,
    do_sample=False,
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=processor.tokenizer.pad_token_id
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

In [5]:
source_path = DATA / "source_rows.json"

if not source_path.exists():
    response = requests.get(
        "https://datasets-server.huggingface.co/rows",
        params={
            "dataset": "lmms-lab-encoder/RefCOCO",
            "config": "default",
            "split": "val",
            "offset": 0,
            "length": 20
        },
        timeout=30
    )
    response.raise_for_status()
    source_path.write_text(
        json.dumps(response.json(), ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

source = json.loads(source_path.read_text(encoding="utf-8"))
print(len(source))

5


In [6]:
examples = []
for item in source["rows"]:
    row = item["row"]
    match = re.match(
        r"(COCO_(train2014|val2014)_\d{12})",
        row["file_name"]   
    )
    
    filename = match.group(1) + ".jpg"
    split_folder = match.group(2)
    image_path = IMAGES / filename
    
    if not image_path.exists():
        url = f"http://images.cocodataset.org/{split_folder}/{filename}"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        image_path.write_bytes(response.content)
    
    with Image.open(image_path) as image:
        width, height = image.size
    
    x, y, w, h = map(float, row["bbox"])
    examples.append({
        "id": row["question_id"],
        "phrase": row["answer"][0],
        "image_path": str(image_path),
        "width": width,
        "height": height,
        "gt_xyxy": [x, y, x + w, y + h],
    })

(DATA / "manifest.json").write_text(
    json.dumps(examples, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Examples:", len(examples))
print("Unique images:", len({e["image_path"] for e in examples}))
print("First phrase:", examples[0]["phrase"])

Examples: 20
Unique images: 9
First phrase: bowl behind the others can only see part
